# Custom CNN training
This notebook is the executable end-to-end training/evaluation module. It uses only the training split for augmentation and class weights; validation drives callbacks; test is evaluated once after model selection.


In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Resolve paths from either the repository root or the notebooks directory.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET = ROOT / "PlantVillage"
OUT = ROOT / "results" / "custom_cnn"
MODEL_DIR = ROOT / "models"
OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.001
RANDOM_SEED = 42
USE_CLASS_WEIGHTS = True

tf.keras.utils.set_random_seed(RANDOM_SEED)

# Run notebooks/02_data_preprocessing.ipynb first to create this split file.
split_path = OUT / "dataset_split.csv"
if not split_path.exists():
    raise FileNotFoundError(
        f"{split_path} was not found. Run notebooks/02_data_preprocessing.ipynb first."
    )

meta = pd.read_csv(split_path)
classes = sorted(meta["class_name"].unique())
NUM_CLASSES = len(classes)

augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomBrightness(0.08),
    ],
    name="training_augmentation",
)


def make_dataset(frame, training=False):
    """Load image paths and integer labels into a batched TensorFlow dataset."""
    dataset = tf.data.Dataset.from_tensor_slices(
        (frame["filepath"].to_numpy(), frame["class_index"].to_numpy())
    )

    def load_image(filepath, label):
        image_bytes = tf.io.read_file(filepath)
        image = tf.image.decode_image(
            image_bytes,
            channels=3,
            expand_animations=False,
        )
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMAGE_SIZE)
        image = tf.cast(image, tf.float32) / 255.0

        if training:
            image = augmentation(image, training=True)

        return image, label

    return (
        dataset
        .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )


train_meta = meta[meta["split"] == "train"]
valid_meta = meta[meta["split"] == "validation"]
test_meta = meta[meta["split"] == "test"]

train = make_dataset(train_meta, training=True)
valid = make_dataset(valid_meta)
test = make_dataset(test_meta)

model = tf.keras.Sequential(
    [
        tf.keras.Input(shape=(*IMAGE_SIZE, 3)),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="custom_cnn",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

class_ids = np.arange(NUM_CLASSES)
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=class_ids,
    y=train_meta["class_index"].to_numpy(),
)
class_weights = dict(zip(class_ids, class_weight_values))

pd.DataFrame(
    {"class_name": classes, "weight": class_weight_values}
).to_csv(OUT / "class_weights.csv", index=False)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_DIR / "custom_cnn_best.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=2,
    ),
]

start_time = time.time()
history = model.fit(
    train,
    validation_data=valid,
    epochs=EPOCHS,
    class_weight=class_weights if USE_CLASS_WEIGHTS else None,
    callbacks=callbacks,
)
training_time = time.time() - start_time

pd.DataFrame(history.history).to_csv(OUT / "training_history.csv", index=False)

for metric, title, filename in [
    ("accuracy", "Accuracy", "accuracy_curve.png"),
    ("loss", "Loss", "loss_curve.png"),
]:
    plt.figure()
    plt.plot(history.history[metric], label="Training")
    plt.plot(history.history[f"val_{metric}"], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT / filename)
    plt.close()

# Evaluate once on the held-out test split.
probabilities = model.predict(test)
y_pred = probabilities.argmax(axis=1)
y_true = test_meta["class_index"].to_numpy()

report = classification_report(
    y_true,
    y_pred,
    labels=class_ids,
    target_names=classes,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).T.to_csv(OUT / "classification_report.csv")

confusion = confusion_matrix(y_true, y_pred, labels=class_ids)
plt.figure(figsize=(12, 10))
sns.heatmap(
    confusion,
    annot=False,
    xticklabels=classes,
    yticklabels=classes,
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png")
plt.close()

metrics = {
    "Model": "Custom CNN",
    "Test Accuracy": report["accuracy"],
    "Macro Precision": report["macro avg"]["precision"],
    "Macro Recall": report["macro avg"]["recall"],
    "Macro F1": report["macro avg"]["f1-score"],
    "Weighted Precision": report["weighted avg"]["precision"],
    "Weighted Recall": report["weighted avg"]["recall"],
    "Weighted F1": report["weighted avg"]["f1-score"],
}
pd.DataFrame([metrics]).to_csv(OUT / "final_metrics.csv", index=False)

complexity = {
    "total_parameters": model.count_params(),
    "trainable_parameters": sum(
        int(np.prod(variable.shape)) for variable in model.trainable_weights
    ),
    "non_trainable_parameters": sum(
        int(np.prod(variable.shape)) for variable in model.non_trainable_weights
    ),
    "training_time_seconds": training_time,
    "epochs_completed": len(history.history["loss"]),
}
pd.DataFrame([complexity]).to_csv(OUT / "model_complexity.csv", index=False)

experiment_config = {
    "dataset_path": str(DATASET),
    "number_of_classes": NUM_CLASSES,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "optimizer": "Adam",
    "loss": "SparseCategoricalCrossentropy",
    "dropout": 0.5,
    "random_seed": RANDOM_SEED,
    "split_ratio": [0.70, 0.15, 0.15],
    "use_class_weights": USE_CLASS_WEIGHTS,
}
with (OUT / "experiment_config.json").open("w", encoding="utf-8") as config_file:
    json.dump(experiment_config, config_file, indent=2)
